# import library

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent          
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import time, os, csv, json
import cv2
import numpy as np
import mediapipe as mp
from lib.GazeTracking.gaze_tracking import GazeTracking 

# CONFIG

In [7]:
TARGET_FPS = 8
PROCESS_INTERVAL = 1.0 / TARGET_FPS

FACING_YAW_DEG = 30.0
FACING_PITCH_DEG = 20.0
MIN_FACING_FOR_GAZE = 16.0

HYSTERESIS_FRAMES = 8
EMA_ALPHA = 0.45

IOU_THRESH = 0.3
MAX_MISSED_SEC = 2.0

CHEAT_MIN_OFFPOSE_SEC = 1.8
CHEAT_MIN_MULTIFACE_SEC = 0.8
CHEAT_MIN_OUTOFFRAME_SEC = 1.0
CHEAT_MIN_EYESOFF_SEC = 0.8
CHEAT_MIN_EYESMOV_SEC = 0.6

CALIB_SECONDS = 2.0

GT_X_DELTA = 0.22              
GT_Y_DELTA = 0.28                 
GT_SPEED_THR = 0.25              
GT_MIN_EOPEN = 0.20             
GT_SPEED_SMOOTH = 0.5             

HEAD_VEL_SMOOTH = 0.4
HEAD_STATIC_VEL_THR = 35.0

SEGMENTS_TO_CSV = True
SEGMENTS_TO_JSON = True
OUT_DIR = "cheat_outputs"
SESSION_PREFIX = time.strftime("%Y%m%d_%H%M%S")

SHOW_DEBUG = False

LMK_IDX = [33, 263, 1, 61, 291, 199]
MODEL_3D = np.array([
    [-43.3,  32.7, -26.0],
    [ 43.3,  32.7, -26.0],
    [  0.0,   0.0,   0.0],
    [-28.9, -28.9, -24.1],
    [ 28.9, -28.9, -24.1],
    [  0.0, -63.6, -12.5],
], dtype=np.float64)

# UTILS

In [3]:
def now_ts(): return time.time()

def iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    areaB = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    return inter / (areaA + areaB - inter + 1e-6)

def ema(prev, new, alpha=EMA_ALPHA):
    return new if prev is None else (alpha*new + (1-alpha)*prev)

def hhmmss(sec):
    sec = max(0.0, float(sec))
    h = int(sec//3600); m = int((sec%3600)//60); s = sec%60
    return f"{h:02d}:{m:02d}:{s:06.3f}"

def estimate_head_pose_flexible(w, h, lm2d, lm3d_or_none):
    pts2d = np.array([[lm2d[i].x*w, lm2d[i].y*h] for i in LMK_IDX], dtype=np.float64)
    if lm3d_or_none is not None:
        pts3d = np.array([[lm3d_or_none[i].x, lm3d_or_none[i].y, lm3d_or_none[i].z] for i in LMK_IDX], dtype=np.float64)
    else:
        pts3d = MODEL_3D.copy()
    K = np.array([[w,0,w/2],[0,w,h/2],[0,0,1]], dtype=np.float64)
    dist = np.zeros((4,1))
    ok, rvec, _ = cv2.solvePnP(pts3d, pts2d, K, dist, flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok: return None, None, None
    R, _ = cv2.Rodrigues(rvec)
    sy = np.sqrt(R[0,0]**2 + R[1,0]**2)
    if sy < 1e-6:
        pitch = np.degrees(np.arctan2(-R[1,2], R[1,1])); yaw = np.degrees(np.arctan2(-R[2,0], sy)); roll = 0.0
    else:
        pitch = np.degrees(np.arctan2(R[2,1], R[2,2]))
        yaw   = np.degrees(np.arctan2(-R[2,0], sy))
        roll  = np.degrees(np.arctan2(R[1,0], R[0,0]))
    return yaw, pitch, roll

# Class

In [5]:
class GazeAdapter: 
    def __init__(self):
        self.gaze = GazeTracking()
        self.center_x = 0.0
        self.center_y = 0.0
        self.active_calib = True
        self.t0 = None
        self.gx_list, self.gy_list = [], []

    def start_calib(self): 
        self.active_calib = True
        self.t0 = now_ts()
        self.gx_list.clear(); self.gy_list.clear()

    def _roi_from_bbox(self, frame, bbox, margin=0.2): 
        h, w = frame.shape[:2]
        x1, y1, x2, y2 = bbox
        bw, bh = x2-x1, y2-y1
        mx, my = int(bw*margin), int(bh*margin)
        X1 = max(0, x1 - mx); Y1 = max(0, y1 - my)
        X2 = min(w, x2 + mx); Y2 = min(h, y2 + my)
        return frame[Y1:Y2, X1:X2].copy(), (X1, Y1)

    def refresh_on_bbox(self, frame, bbox):  
        roi, (ox, oy) = self._roi_from_bbox(frame, bbox)
        if roi.size == 0: return None
        self.gaze.refresh(roi)
        hx = self.gaze.horizontal_ratio()
        hy = self.gaze.vertical_ratio()
        if hx is None or hy is None:
            return {"gx": None, "gy": None, "blink": self.gaze.is_blinking(),
                    "left": self.gaze.is_left(), "right": self.gaze.is_right(), "center": self.gaze.is_center()}
        gx = float((hx - 0.5) * 2.0)  
        gy = float((hy - 0.5) * 2.0)  
        if self.active_calib:
            self.gx_list.append(gx); self.gy_list.append(gy)
        return {"gx": gx, "gy": gy, "blink": self.gaze.is_blinking(),
                "left": self.gaze.is_left(), "right": self.gaze.is_right(), "center": self.gaze.is_center()}

    def finish_calib_if_ready(self, seconds=CALIB_SECONDS): 
        if not self.active_calib: return False
        if self.t0 is None or (now_ts() - self.t0) < seconds: return False
        if self.gx_list:
            self.center_x = float(np.median(self.gx_list))
        if self.gy_list:
            self.center_y = float(np.median(self.gy_list))
        self.active_calib = False
        return True

class HeadCalib:
    def __init__(self):
        self.active = True
        self.t0 = None
        self.yaw_list = []; self.pitch_list = []
        self.bias_yaw = 0.0; self.bias_pitch = 0.0

    def start(self):
        self.active = True; self.t0 = now_ts()
        self.yaw_list.clear(); self.pitch_list.clear()

    def feed(self, yaw, pitch):
        if yaw is not None and pitch is not None:
            self.yaw_list.append(yaw); self.pitch_list.append(pitch)

    def done_if_ready(self):
        if not self.active or (now_ts() - self.t0) < CALIB_SECONDS: return False
        if self.yaw_list:   self.bias_yaw = float(np.median(self.yaw_list))
        if self.pitch_list: self.bias_pitch = float(np.median(self.pitch_list))
        self.active = False
        return True

class Track:
    _next_id = 1

    def __init__(self, bbox):
        self.id = Track._next_id; Track._next_id += 1
        self.bbox = bbox
        self.ts_last = now_ts()
        self.yaw_s = None; self.pitch_s = None; self.roll_s = None
        self.prev_yaw = None; self.prev_pitch = None; self.prev_ang_t = None
        self.head_vel = None
        self.facing_prob = None
        self.state = "UNKNOWN"
        self._up = 0; self._dn = 0
        self.ts_not_focus_start = None
        self.gx_s = None; self.gy_s = None
        self.prev_gx = None; self.prev_gy = None; self.prev_t = None
        self.gaze_speed = None
        self.ts_eyes_off_start = None
        self.ts_eyes_moving_start = None
        self.flags = {"HEAD_POSE_OFF":False, "EYES_OFF":False, "EYES_MOVING":False, "OUT_OF_FRAME":False}
        self.cheat_active = False
        self.cheat_reason = None
        self.ts_cheat_last = None

    def update_pose(self, yaw, pitch, roll):
        self.yaw_s = ema(self.yaw_s, yaw)
        self.pitch_s = ema(self.pitch_s, pitch)
        self.roll_s = ema(self.roll_s, roll)
        t = now_ts()

        if self.prev_yaw is not None and self.prev_pitch is not None and self.prev_ang_t is not None:
            dt = max(1e-3, t - self.prev_ang_t)
            dy = (self.yaw_s - self.prev_yaw) if self.yaw_s is not None else 0.0
            dp = (self.pitch_s - self.prev_pitch) if self.pitch_s is not None else 0.0
            vel = np.hypot(dy, dp) / dt
            self.head_vel = ema(self.head_vel, vel, alpha=HEAD_VEL_SMOOTH)
        self.prev_yaw, self.prev_pitch, self.prev_ang_t = self.yaw_s, self.pitch_s, t

    def update_bbox(self, bbox):
        self.bbox = bbox; self.ts_last = now_ts()

    def update_facing(self, facing_now):
        p = 1.0 if facing_now else 0.0
        self.facing_prob = ema(self.facing_prob, p)

        if facing_now: self._up += 1; self._dn = 0

        else: self._dn += 1; self._up = 0

        if self.state in ("UNKNOWN","NOT_FOCUS"):
            if self._up >= HYSTERESIS_FRAMES and (self.facing_prob or 0) >= 0.6:
                self.state = "FOCUS"

        if self.state in ("UNKNOWN","FOCUS"):
            if self._dn >= HYSTERESIS_FRAMES and (self.facing_prob or 1) <= 0.4:
                self.state = "NOT_FOCUS"

        t = now_ts()

        if self.state == "NOT_FOCUS":
            if self.ts_not_focus_start is None: self.ts_not_focus_start = t

        else:
            self.ts_not_focus_start = None

    def update_gaze(self, gx, gy, allow_eval):
        if gx is None or gy is None:
            self.ts_eyes_off_start = None
            self.ts_eyes_moving_start = None
            self.prev_gx = gx; self.prev_gy = gy; self.prev_t = now_ts()
            return
        
        self.gx_s = ema(self.gx_s, gx)
        self.gy_s = ema(self.gy_s, gy)
        t = now_ts()

        if self.prev_gx is not None and self.prev_gy is not None and self.prev_t is not None:
            dt = max(1e-3, t - self.prev_t)
            dx = self.gx_s - self.prev_gx
            dy = self.gy_s - self.prev_gy
            spd = (dx**2 + dy**2)**0.5 / dt
            self.gaze_speed = ema(self.gaze_speed, spd, alpha=GT_SPEED_SMOOTH)
        self.prev_gx = self.gx_s; self.prev_gy = self.gy_s; self.prev_t = t

        if not allow_eval:
            self.ts_eyes_off_start = None
            self.ts_eyes_moving_start = None
            return
        
        devx = abs(self.gx_s)
        devy = abs(self.gy_s)
        off = (devx > GT_X_DELTA) or (devy > GT_Y_DELTA)

        if off:
            if self.ts_eyes_off_start is None: self.ts_eyes_off_start = t
        else:
            self.ts_eyes_off_start = None

        head_static = (self.head_vel or 0.0) < HEAD_STATIC_VEL_THR
        fast = (self.gaze_speed or 0.0) > GT_SPEED_THR
        moving = head_static and fast

        if moving:
            if self.ts_eyes_moving_start is None: self.ts_eyes_moving_start = t
        else:
            self.ts_eyes_moving_start = None

    def mark_flag(self, key, val): self.flags[key] = val

    def mark_cheat(self, reason):
        self.cheat_active = True; self.cheat_reason = reason; self.ts_cheat_last = now_ts()

    def clear_cheat(self):
        self.cheat_active = False; self.cheat_reason = None; self.ts_cheat_last = now_ts()

class Tracker:
    def __init__(self): self.tracks = []

    def update(self, bboxes):
        assigned = set()

        for t in self.tracks:
            best, idx = 0.0, -1

            for j, b in enumerate(bboxes):

                if j in assigned: continue
                i = iou(t.bbox, b)

                if i > best: best, idx = i, j

            if best >= IOU_THRESH and idx >= 0:
                t.update_bbox(bboxes[idx]); assigned.add(idx)

        for j, b in enumerate(bboxes):
            if j not in assigned: self.tracks.append(Track(b))

        tnow = now_ts()
        self.tracks = [t for t in self.tracks if (tnow - t.ts_last) <= MAX_MISSED_SEC]
        return self.tracks

class SegmentLogger:
    def __init__(self, out_dir, prefix, session_start):
        self.out_dir = out_dir; os.makedirs(out_dir, exist_ok=True)
        self.prefix = prefix; self.session_start = session_start
        self.open_segments = {}
        self.closed_segments = []
        self.csv_path = os.path.join(out_dir, f"{prefix}_segments.csv")
        self.json_path = os.path.join(out_dir, f"{prefix}_segments.json")

        if SEGMENTS_TO_CSV and not os.path.exists(self.csv_path):
            with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["track_id","reason","start_ts","end_ts","duration_sec","start_hhmmss","end_hhmmss"])

    def _append_closed(self, seg):
        self.closed_segments.append(seg)
        if SEGMENTS_TO_CSV:
            with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow([seg["track_id"], seg["reason"],
                            f'{seg["start_ts"]:.3f}', f'{seg["end_ts"]:.3f}',
                            f'{seg["duration_sec"]:.3f}',
                            seg["start_hhmmss"], seg["end_hhmmss"]])
                
    def update_track(self, track: Track, current_reason: str | None):
        tid = track.id; tnow = now_ts()

        if tid in self.open_segments:
            seg = self.open_segments[tid]

            if current_reason != seg["reason"]:
                seg["end_ts"] = tnow
                seg["duration_sec"] = seg["end_ts"] - seg["start_ts"]
                seg["start_hhmmss"] = hhmmss(seg["start_ts"] - self.session_start)
                seg["end_hhmmss"] = hhmmss(seg["end_ts"] - self.session_start)
                self._append_closed(seg)
                del self.open_segments[tid]

        if current_reason is not None and tid not in self.open_segments:
            self.open_segments[tid] = {"track_id": tid, "reason": current_reason, "start_ts": tnow}

    def close_all(self):
        tnow = now_ts()

        for tid, seg in list(self.open_segments.items()):
            seg["end_ts"] = tnow
            seg["duration_sec"] = seg["end_ts"] - seg["start_ts"]
            seg["start_hhmmss"] = hhmmss(seg["start_ts"] - self.session_start)
            seg["end_hhmmss"] = hhmmss(seg["end_ts"] - self.session_start)
            self._append_closed(seg)
            del self.open_segments[tid]

        if SEGMENTS_TO_JSON:
            with open(self.json_path, "w", encoding="utf-8") as f:
                json.dump({"session_start_unix": self.session_start,
                           "session_start_readable": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(self.session_start)),
                           "segments": self.closed_segments}, f, ensure_ascii=False, indent=2)

In [12]:
def main():
    session_start = now_ts()
    seglog = SegmentLogger(OUT_DIR, SESSION_PREFIX, session_start)

    cap = cv2.VideoCapture(0)
    last_proc = 0.0
    tracker = Tracker()

    head_cal = HeadCalib(); head_cal.start()
    gaze_adapt = GazeAdapter(); gaze_adapt.start_calib() 

    mp_fd = mp.solutions.face_detection
    mp_fm = mp.solutions.face_mesh

    with mp_fd.FaceDetection(model_selection=0, min_detection_confidence=0.5) as fd, \
         mp_fm.FaceMesh(max_num_faces=5, refine_landmarks=False, 
                        min_detection_confidence=0.5, min_tracking_confidence=0.5) as fm:

        while True:
            ok, frame = cap.read()
            if not ok: break
            h, w = frame.shape[:2]
            draw = frame.copy()
            tnow = now_ts()
            do_process = (tnow - last_proc) >= PROCESS_INTERVAL

            if do_process:
                last_proc = tnow
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                bboxes = []
                det = fd.process(rgb)
                if det and det.detections:
                    for d in det.detections:
                        rel = d.location_data.relative_bounding_box
                        x1 = int(rel.xmin*w); y1 = int(rel.ymin*h)
                        x2 = int((rel.xmin+rel.width)*w); y2 = int((rel.ymin+rel.height)*h)
                        x1 = max(0,x1); y1 = max(0,y1); x2 = min(w-1,x2); y2 = min(h-1,y2)
                        if x2>x1 and y2>y1: bboxes.append([x1,y1,x2,y2])

                tracks = tracker.update(bboxes)
                mesh_res = fm.process(rgb)

                lmk2d_list = []
                lmk3d_list = None
                if mesh_res and mesh_res.multi_face_landmarks:
                    lmk2d_list = mesh_res.multi_face_landmarks
                    has_world = hasattr(mesh_res,"multi_face_world_landmarks") and (mesh_res.multi_face_world_landmarks is not None)
                    lmk3d_list = mesh_res.multi_face_world_landmarks if has_world else [None]*len(lmk2d_list)

                    centers = []
                    for lm2d in lmk2d_list:
                        nose = lm2d.landmark[1]
                        centers.append((nose.x*w, nose.y*h))

                    for t in tracks:
                        cx = (t.bbox[0]+t.bbox[2])/2.0
                        cy = (t.bbox[1]+t.bbox[3])/2.0
                        if not centers: continue
                        idx = int(np.argmin([(cx-x)**2 + (cy-y)**2 for (x,y) in centers]))
                        lm2d = lmk2d_list[idx].landmark
                        lm3d = None if (lmk3d_list is None or lmk3d_list[idx] is None) else lmk3d_list[idx].landmark

                        yaw, pitch, roll = estimate_head_pose_flexible(w, h, lm2d, lm3d)
                        yaw_p = None if yaw is None else (yaw - head_cal.bias_yaw)
                        pit_p = None if pitch is None else (pitch - head_cal.bias_pitch)
                        t.update_pose(yaw_p, pit_p, roll)

                        head_cal.feed(yaw, pitch)
                        head_cal.done_if_ready()

                        facing_now = (t.yaw_s is not None and t.pitch_s is not None and
                                      abs(t.yaw_s) <= FACING_YAW_DEG and abs(t.pitch_s) <= FACING_PITCH_DEG)
                        t.update_facing(facing_now)

                        allow_eval_gaze = (not gaze_adapt.active_calib) and facing_now and \
                                          (abs(t.yaw_s) <= MIN_FACING_FOR_GAZE and abs(t.pitch_s) <= MIN_FACING_FOR_GAZE)

                        g = gaze_adapt.refresh_on_bbox(frame, t.bbox)
                        if g is not None:
                            gx = None if g["gx"] is None else (g["gx"] - gaze_adapt.center_x)  
                            gy = None if g["gy"] is None else (g["gy"] - gaze_adapt.center_y) 
                            t.update_gaze(gx, gy, allow_eval_gaze)  

                        gaze_adapt.finish_calib_if_ready()  

                        if not gaze_adapt.active_calib:
                            nowt = now_ts()
                            head_offset = (abs(t.yaw_s or 0) > FACING_YAW_DEG) or (abs(t.pitch_s or 0) > FACING_PITCH_DEG)
                            head_quiet  = (t.head_vel or 0.0) < HEAD_STATIC_VEL_THR
                            head_off_ready = head_offset and head_quiet and (t.ts_not_focus_start is not None and (nowt - t.ts_not_focus_start) >= CHEAT_MIN_OFFPOSE_SEC)
                            t.mark_flag("HEAD_POSE_OFF", head_off_ready)
                            t.mark_flag("EYES_OFF",      t.ts_eyes_off_start is not None and (nowt - t.ts_eyes_off_start) >= CHEAT_MIN_EYESOFF_SEC)
                            t.mark_flag("EYES_MOVING",   t.ts_eyes_moving_start is not None and (nowt - t.ts_eyes_moving_start) >= CHEAT_MIN_EYESMOV_SEC)
                            t.mark_flag("OUT_OF_FRAME",  (nowt - t.ts_last) >= CHEAT_MIN_OUTOFFRAME_SEC)

                multi_faces_flag = (len(tracker.tracks) > 1) and (not gaze_adapt.active_calib)
                if multi_faces_flag:
                    oldest_seen = min(t.ts_last for t in tracker.tracks) if tracker.tracks else now_ts()
                    multi_faces_flag = (tnow - oldest_seen) >= CHEAT_MIN_MULTIFACE_SEC

                for t in tracker.tracks:
                    current_reason = None
                    if multi_faces_flag: current_reason = "MULTIPLE_FACES"
                    elif t.flags["OUT_OF_FRAME"]: current_reason = "OUT_OF_FRAME"
                    elif t.flags["EYES_OFF"]: current_reason = "EYES_OFF"
                    elif t.flags["EYES_MOVING"]: current_reason = "EYES_MOVING"
                    elif t.flags["HEAD_POSE_OFF"]: current_reason = "HEAD_POSE_OFF"
                    if gaze_adapt.active_calib: current_reason = None  
                    prev_reason = t.cheat_reason if t.cheat_active else None
                    if current_reason != prev_reason:
                        if current_reason is None and t.cheat_active:
                            t.clear_cheat()
                        elif current_reason is not None:
                            t.mark_cheat(current_reason)
                    seglog.update_track(t, current_reason)

            for t in tracker.tracks:
                x1,y1,x2,y2 = map(int, t.bbox)
                base = (0,200,0) if t.state=="FOCUS" else (0,0,200)
                color = (0,0,255) if t.cheat_active else base
                cv2.rectangle(draw, (x1,y1), (x2,y2), color, 2)
                label = f"ID {t.id} | {t.state}"

                if SHOW_DEBUG and t.gaze_speed is not None:
                    label += f" | v {t.gaze_speed:.2f}"

                if t.cheat_active and t.cheat_reason:
                    label += f" | CHEATING:{t.cheat_reason}"

                cv2.putText(draw, label, (x1, max(20, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.50, color, 2)

            if len(tracker.tracks) > 1:
                cv2.putText(draw, f"MULTIPLE FACES: {len(tracker.tracks)}", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

            if SHOW_DEBUG:
                st = "CALIB" if gaze_adapt.active_calib else "RUN"
                cv2.putText(draw, f"{st} | thr yaw<=±{int(FACING_YAW_DEG)}, pitch<=±{int(FACING_PITCH_DEG)}", (10, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
                
                cv2.putText(draw, f"Proc ~{TARGET_FPS} FPS | Tracks:{len(tracker.tracks)}", (10, h-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

            cv2.imshow("Cheating Detection with GazeTracking", draw)
            if cv2.waitKey(1) & 0xFF == 27: break

    seglog.close_all()
    cap.release(); cv2.destroyAllWindows()

In [13]:
main()